## **AIM**
Locard exchange principle in python

In [1]:
import os
import time
import hashlib
from datetime import datetime


def fmt(ts):
    return datetime.fromtimestamp(ts).strftime("%H:%M:%S.%f")[:-3]


def snapshot(path):
    st = os.stat(path)
    with open(path, "rb") as f:
        digest = hashlib.sha256(f.read()).hexdigest()
    return {
        "size": st.st_size,
        "atime": fmt(st.st_atime),
        "mtime": fmt(st.st_mtime),
        "ctime": fmt(st.st_ctime),
        "sha256": digest,
    }


def diff(label, before, after):
    print(f"\n{label}")
    for key in before:
        if before[key] != after[key]:
            print(f"  {key}: {before[key]} -> {after[key]}")
        else:
            print(f"  {key}: {before[key]} -> {after[key]} (unchanged)")


def main():
    path = "evidence.txt"
    open(path, "w").write("original content\n")
    snapshots = [snapshot(path)]

    time.sleep(1)
    open(path).read()
    os.utime(path, (time.time(), os.stat(path).st_mtime))
    snapshots.append(snapshot(path))

    time.sleep(1)
    open(path, "a").write("tampered line\n")
    snapshots.append(snapshot(path))

    time.sleep(1)
    renamed = "evidence_moved.txt"
    os.rename(path, renamed)
    snapshots.append(snapshot(renamed))

    labels = ["create -> read", "read -> modify", "modify -> rename"]
    for label, before, after in zip(labels, snapshots, snapshots[1:]):
        diff(label, before, after)


if __name__ == "__main__":
    main()



create -> read
  size: 17 -> 17 (unchanged)
  atime: 08:01:22.985 -> 08:01:23.987
  mtime: 08:01:22.986 -> 08:01:22.986 (unchanged)
  ctime: 08:01:22.986 -> 08:01:23.986
  sha256: 516ad7b388b21e05e8c56229f063d112e70a2fea45fdd357e8ff44e6a5bce689 -> 516ad7b388b21e05e8c56229f063d112e70a2fea45fdd357e8ff44e6a5bce689 (unchanged)

read -> modify
  size: 17 -> 31
  atime: 08:01:23.987 -> 08:01:23.987 (unchanged)
  mtime: 08:01:22.986 -> 08:01:24.987
  ctime: 08:01:23.986 -> 08:01:24.987
  sha256: 516ad7b388b21e05e8c56229f063d112e70a2fea45fdd357e8ff44e6a5bce689 -> 680e794e736131f2d2dd6ad810d3a493f8bec0fc8d5a639ad70e0733da183ae2

modify -> rename
  size: 31 -> 31 (unchanged)
  atime: 08:01:23.987 -> 08:01:24.987
  mtime: 08:01:24.987 -> 08:01:24.987 (unchanged)
  ctime: 08:01:24.987 -> 08:01:25.987
  sha256: 680e794e736131f2d2dd6ad810d3a493f8bec0fc8d5a639ad70e0733da183ae2 -> 680e794e736131f2d2dd6ad810d3a493f8bec0fc8d5a639ad70e0733da183ae2 (unchanged)
